# The diagonal method: does traffic travel along characteristics?

**An idea from traffic physics, tested.** In the space-time diagram of a road (position across, time down) the classical theory of traffic flow (Lighthill-Whitham-Richards, LWR) says that
information travels along straight lines, *characteristics*: cars in free flow move along steep diagonals (~115 km/h), and the back of a jam moves **backwards** along a shallower one
(~25 km/h upstream). So a forecast could be built from geometry: **estimate the 5-minute shift per block, slide the last observed row along that diagonal to fill the unknown triangle, and let a model fill what the diagonals do not reach.**
We tested exactly that on the TrafficFlowBench data - the premise first, then the forecast, then the two tasks.

**Result, with the numbers below:**
* On these corridors the speed field is **not carried along any diagonal**: predicting the next 5-minute row from the current one shifted by *any* distance is best at a shift of **0.0 km**, in free flow and next to a jam, on both corridors tested. Next to a jam, on D12_I5_N, the error at zero shift is 0.037 (as a share of free-flow speed), at the free-flow characteristic 0.48 (13×), at the jam-wave characteristic 0.26 (7×); on D7_I405_N 0.035, 0.37 and 0.12.
* The back of a jam does not travel like a wave. The textbook wave from the released fundamental diagram is **24 km/h**; the measured upstream speed of the tail has a **95th percentile of 7.6 km/h** (median 0), and on the example day the tail moved 3.3 km in three hours where the wave would have moved 77 km.
* As a forecast of the jam in the next 30 minutes (Task 2) the diagonal tail extrapolation scores **0.658 IoU against 0.801 for a gradient-boosted model** (−0.124 [−0.156, −0.096], 159 windows); every combination of the two is worse than the model alone.
* In Task 1 (filling blanked cells) the same idea is 3-10 times worse than 'the same link five minutes earlier', and as a model feature it adds nothing; the one spatial feature of this family that *did* help is a different one (section 5).

**Caveat that matters:** this is **synthetic data from a generator**, not recorded traffic. The result says the generator's jams do not behave as LWR waves; it says nothing about real freeways, where the theory was designed for.

In [ ]:
import os, glob, numpy as np, pandas as pd
from pathlib import Path
KIN = Path(os.environ.get('KIN', '/kaggle/input'))
root = [p for depth in range(1, 6) for p in KIN.glob('/'.join(['*'] * depth) + '/corridors') if (p / 'D12_I5_N').exists()][0]
def corridor(panel):
    """links in driving order, x_km along the direction of travel, free-flow speed and length per link"""
    topo = pd.read_csv(f'{root}/{panel}/network/lwr_mainline_topology.csv'); fd = pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv').set_index('link_id')
    f = sorted(glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_*.parquet'))[0]
    mp = pd.read_parquet(f, columns=['link_id', 'milepost']).groupby('link_id').milepost.mean(); ml = set(topo.link_id)
    s = [np.sign(mp[o] - mp[r.link_id]) for r in topo.itertuples() for o in str(r.outgoing_link_ids).split(';') if o in ml and r.link_id in mp.index and o in mp.index]
    sgn = 1.0 if np.sum(s) >= 0 else -1.0
    order = (mp[mp.index.isin(ml)] * sgn).sort_values().index.tolist(); x = mp.reindex(order).to_numpy() * sgn
    return dict(order=order, x=x - x.min(), vf=fd.loc[order, 'free_speed_kmh'].to_numpy(), length=fd.loc[order, 'length_km'].to_numpy(), cap=fd.loc[order, 'capacity_vph'].to_numpy(), lanes=fd.loc[order, 'lanes'].to_numpy())
def day_grid(panel, date, order, col='speed_kmh'):
    """(288, L) grid of one train day; NaN where the cell is not score-eligible (not observed)"""
    f = glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_{date.replace("-", "_")}.parquet')[0]
    d = pd.read_parquet(f, columns=['timestamp', 'link_id', col, 'is_score_eligible']); d = d[d.is_score_eligible.astype(bool)]
    li = {l: i for i, l in enumerate(order)}; t = pd.to_datetime(d.timestamp); G = np.full((288, len(order)), np.nan)
    G[(t.dt.hour * 12 + t.dt.minute // 5).to_numpy(), d.link_id.map(li).to_numpy()] = d[col].to_numpy(); return G
import matplotlib.pyplot as plt
print('competition data at', root)

## 1. What the released physics says

The competition ships a fundamental diagram per link (`fd_parameters.csv`): free-flow speed, capacity, critical density, jam density. For a triangular diagram the wave speed in congestion is
`w = capacity / (jam density − critical density)`. With a record every 5 minutes, one step is

* **free flow:** 113-118 km/h ≈ **9.5 km** per step, i.e. 24-94 links away (links are 0.1-0.4 km long),
* **congestion:** 23-28 km/h backwards ≈ **2 km** per step, i.e. 5-19 links.

In [ ]:
# characteristic speeds from the released fundamental-diagram parameters
import matplotlib.pyplot as plt, time
panels = sorted(p.name for p in Path(root).iterdir() if p.is_dir() and (p / 'train').exists()); rows = []
for panel in panels:
    fd = pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv'); w = fd.capacity_vph / (fd.k_jam - fd.critical_density)          # triangular FD: wave speed = capacity / (jam density - critical density)
    rows.append(dict(corridor=panel, free_flow_kmh=fd.free_speed_kmh.median(), wave_kmh=w.median(), link_km=fd.length_km.median(), free_links_per_5min=fd.free_speed_kmh.median() * 5 / 60 / fd.length_km.median(), wave_links_per_5min=w.median() * 5 / 60 / fd.length_km.median()))
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
FD = pd.DataFrame(rows).set_index('corridor').round(1); FD

## 2. Test 1 - is the field carried along a diagonal?

The cleanest test of the premise needs no forecasting model: predict the **next row** (all links, 5 minutes later, as share of free-flow speed) from the **current row shifted by a distance Δ** along the road,
and look at the error as a function of Δ. If traffic travelled along characteristics the minimum would sit near the free-flow shift (+9.5 km) in free flow and near the wave shift (−2 km) in a jam.
12 days of January 2031 on two corridors, every 5-minute pair of rows, observed cells only; 'free' = no cell below 0.9 × free-flow speed within 1 km of the target, 'next to a jam' = a cell below 0.6 × free-flow speed within 1 km (0.6 is the competition's own queue threshold).

In [ ]:
# test 1: is the field 'carried' along a diagonal? predict the next row from this row shifted by a distance
deltas = np.arange(-12, 12.01, 0.25)                                         # km moved in one 5-minute step (>0 = with the traffic)
def shift_curve(panel, dates):
    c = corridor(panel); x, vf = c['x'], c['vf']; near = np.abs(x[:, None] - x[None, :]) <= 1.0; err = {r: np.zeros(len(deltas)) for r in ('free (no cell < 0.9 within 1 km)', 'next to a jam (a cell < 0.6 within 1 km)')}; cnt = {r: np.zeros(len(deltas)) for r in err}
    for d in dates:
        G = day_grid(panel, d, c['order']) / vf
        for t in range(287):
            a, b = G[t], G[t + 1]; ok = np.isfinite(a)
            if ok.sum() < 30: continue
            af = np.interp(x, x[ok], a[ok]); tgt = np.isfinite(b); jamnear = (near & (af < 0.6)[None, :]).any(1); freeok = ~(near & (af < 0.9)[None, :]).any(1)
            for i, dl in enumerate(deltas):
                pred = np.interp(x - dl, x, af); inside = (x - dl >= x[0]) & (x - dl <= x[-1]) & tgt
                for r, m in zip(err, (freeok, jamnear)): s = inside & m; err[r][i] += np.abs(pred[s] - b[s]).sum(); cnt[r][i] += s.sum()
    return {r: err[r] / np.maximum(cnt[r], 1) for r in err}, {r: int(cnt[r][len(deltas) // 2]) for r in err}, c
DATES = ['2031-01-%02d' % k for k in (5, 7, 9, 12, 14, 16, 19, 21, 23, 26, 28, 30)]; t0 = time.time(); fig, axs = plt.subplots(1, 2, figsize=(13, 3.8)); best = {}; summ = []
for ax, panel in zip(axs, ('D12_I5_N', 'D7_I405_N')):
    curves, n0, c = shift_curve(panel, DATES); fd = pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv'); vfree, w = fd.free_speed_kmh.median(), (fd.capacity_vph / (fd.k_jam - fd.critical_density)).median()
    at = lambda y, d: float(y[np.argmin(np.abs(deltas - d))])
    for (r, y), col in zip(curves.items(), ('#2a6fb0', '#b3261e')): ax.plot(deltas, y, color=col, label=f'{r}  (n={n0[r]:,})'); best[(panel, r)] = float(deltas[np.argmin(y)]); summ.append(dict(corridor=panel, regime=r.split(' (')[0], mae_no_shift=at(y, 0), mae_free_shift=at(y, vfree * 5 / 60), mae_wave_shift=at(y, -w * 5 / 60), best_shift_km=best[(panel, r)]))
    ax.axvline(vfree * 5 / 60, color='k', ls='--', lw=0.8); ax.axvline(-w * 5 / 60, color='k', ls=':', lw=0.8); ax.text(vfree * 5 / 60 - 0.2, ax.get_ylim()[1] * 0.5, 'free-flow characteristic', fontsize=7, rotation=90, ha='right', va='center'); ax.text(-w * 5 / 60 - 0.2, ax.get_ylim()[1] * 0.5, 'jam-wave characteristic', fontsize=7, rotation=90, ha='right', va='center')
    ax.set_xlabel('shift of the previous 5-minute row, km (with the traffic →)'); ax.set_ylabel('MAE of v / v_free, next row'); ax.set_title(panel, fontsize=10); ax.legend(fontsize=7, frameon=False, loc='upper center')
plt.tight_layout(); plt.show(); print(f'{time.time() - t0:.0f} s')
pd.DataFrame(summ).set_index(['corridor', 'regime']).round(3)

**Both regimes, both corridors: the best shift is 0.0 km.** The dotted and dashed lines mark the two theoretical characteristics; the data are nowhere near them. In free flow the errors are small everywhere (0.01-0.05) and grow with |Δ| because each link has its own level (on D7_I405_N the curve is almost flat: 0.018 without shift, 0.022 at the free-flow shift). Next to a jam
the error at the free-flow characteristic is 11-13 times the error with no shift at all.

*(Is the zero an artefact?* An exact zero deserves suspicion. Here it is a sharp V on a 0.25 km grid with a different shape on each side, the shift code is the same for every Δ, and the two corridors have very different link spacing - the minimum sits at 0 for both.)

## 3. Test 2 - the one diagonal that should move: the back of the jam

If nothing travels with the cars, the other diagonal is the jam's upstream edge. We follow the upstream edge of the longest jam run on four corridors and measure its displacement over one hour (positive = upstream, i.e. the jam is growing):

In [ ]:
# test 2: how fast does the tail of a jam actually move?
def runs(mask):
    out, j, L = [], 0, len(mask)
    while j < L:
        if mask[j]:
            k = j
            while k + 1 < L and mask[k + 1]: k += 1
            out.append((j, k)); j = k + 1
        else: j += 1
    return out
def tail_track(panel, date):
    c = corridor(panel); x, vf = c['x'], c['vf']; G = day_grid(panel, date, c['order']) / vf; edge = np.full(288, np.nan); lo = np.full(288, -1); hi = np.full(288, -1)
    for t in range(288):
        a = G[t]; ok = np.isfinite(a)
        if ok.sum() < 30: continue
        af = np.interp(x, x[ok], a[ok]); r = [q for q in runs(af < 0.6) if q[1] - q[0] >= 2]
        if r: q = max(r, key=lambda q: q[1] - q[0]); edge[t] = x[q[0]]; lo[t], hi[t] = q
    return x, edge, lo, hi, G
speeds = []
for panel in ('D12_I5_N', 'D7_I405_N', 'D12_I5_S', 'D7_I210_W'):
    for d in DATES:
        x, edge, lo, hi, _ = tail_track(panel, d)
        for t in range(0, 276):                                               # 1-hour differences of the upstream edge of the longest jam, only while it is the same jam (overlapping runs)
            if np.isfinite(edge[t]) and np.isfinite(edge[t + 12]) and lo[t + 12] <= hi[t] and lo[t] <= hi[t + 12]: speeds.append(edge[t] - edge[t + 12])
sp = np.array(speeds); fdw = np.median([(pd.read_csv(f'{root}/{p}/network/fd_parameters.csv').pipe(lambda f: (f.capacity_vph / (f.k_jam - f.critical_density)).median())) for p in ('D12_I5_N', 'D7_I405_N')])
print(f'{len(sp):,} one-hour tail displacements (upstream = positive).  median {np.median(sp):+.1f} km/h, interquartile {np.percentile(sp, 25):+.1f} … {np.percentile(sp, 75):+.1f}, 95th percentile {np.percentile(sp, 95):+.1f} km/h;  textbook jam wave: {fdw:.0f} km/h')

The median is zero (a jam sitting on a bottleneck, or the structural block at the corridor entrance, does not move); the **95th percentile is +7.6 km/h, a third of the textbook 24 km/h.** The picture below follows one jam on one day. Black: its upstream edge. Magenta dashed: the textbook wave starting at the same point. Cyan dotted: the free-flow characteristic.

In [ ]:
# the picture: the textbook wave against the measured tail
def follow(panel, date, x0, hour0):                                           # follow the jam that contains km x0 at hour0: each step the run that overlaps the previous one
    c = corridor(panel); x, vf = c['x'], c['vf']; G = day_grid(panel, date, c['order']) / vf; edge = np.full(288, np.nan); prev, lost = None, 0
    for t in range(int(hour0 * 12), 288):
        a = G[t]; ok = np.isfinite(a)
        if ok.sum() < 30: continue
        r = [q for q in runs(np.interp(x, x[ok], a[ok]) < 0.6) if q[1] - q[0] >= 2]
        cand = [q for q in r if x[q[0]] <= x0 <= x[q[1]]] if prev is None else [q for q in r if q[0] <= prev[1] and prev[0] <= q[1]]
        if not cand:
            lost += 1
            if prev is not None and lost > 6: break
            continue
        prev = max(cand, key=lambda q: x[q[1]] - x[q[0]]); edge[t] = x[prev[0]]; lost = 0
    return x, edge, G, c
panel, date = 'D12_I5_N', '2031-01-23'; x, edge, G, c = follow(panel, date, 33.0, 6.5); edges = np.r_[x[0] - .1, (x[1:] + x[:-1]) / 2, x[-1] + .1]
cm = plt.get_cmap('RdYlGn').copy(); cm.set_bad('#d9d9d9'); fig, ax = plt.subplots(figsize=(9, 6)); ax.pcolormesh(edges, np.r_[np.arange(288) / 12, 24], G * c['vf'], cmap=cm, vmin=0, vmax=120, shading='flat'); ax.invert_yaxis()
t_start = int(np.flatnonzero(np.isfinite(edge))[0]); w = float(np.median(pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv').pipe(lambda f: f.capacity_vph / (f.k_jam - f.critical_density)))); hrs = np.arange(0, 3.01, 0.25)
ax.plot(edge, np.arange(288) / 12, 'k', lw=2, label='measured upstream edge of this jam'); ax.plot(edge[t_start] - w * hrs, t_start / 12 + hrs, 'm--', lw=2, label=f'textbook jam wave: {w:.0f} km/h upstream from the same start')
ax.plot(edge[t_start] + 110 * hrs / 12, t_start / 12 + hrs, 'c:', lw=2, label='free-flow characteristic: +110 km/h'); ax.set_xlim(x[0], x[-1]); ax.set_xlabel('km along direction of travel →'); ax.set_ylabel('hour'); ax.legend(fontsize=8, loc='lower right', framealpha=0.9); ax.set_title(f'{panel} {date}: speed (km/h)'); plt.show()
t3 = min(t_start + 36, 287); mv = edge[t_start] - edge[t3]; tl = np.flatnonzero(np.isfinite(edge))[-1]
print(f'jam first seen at hour {t_start / 12:.1f}, km {edge[t_start]:.1f}; tracked until hour {tl / 12:.1f}. Three hours after the start the edge has moved {mv:+.1f} km upstream = {mv / 3:.1f} km/h; the textbook wave would have moved {3 * w:.0f} km')
es = pd.Series(edge).rolling(9, center=True, min_periods=5).median().to_numpy(); k = int(np.nanargmin(es))              # smoothed over 45 min so that one-step hops of the tracker do not count
pos = {h: float(es[int(h * 12)]) for h in (8, 10, 12, 15, 18, 21) if np.isfinite(es[int(h * 12)])}; print('edge position (km, 45-min median) at hour:', {h: round(v, 1) for h, v in pos.items()})
print(f'furthest upstream: km {es[k]:.1f} at hour {k / 12:.1f} = {edge[t_start] - es[k]:.1f} km in {(k - t_start) / 12:.1f} h = {(edge[t_start] - es[k]) / ((k - t_start) / 12):.1f} km/h on average while growing')

The tail **creeps**: about 1 km/h in the first three hours (the last printed lines give the edge positions through the day and the average speed while the jam was growing). The wave line leaves the picture within an hour. (The few horizontal black strokes near hours 17-19 are the tracker hopping between neighbouring jam runs, not the jam.)

## 4. What the benchmark said (static numbers from our log)

**Task 2 - forecast the jam 30 minutes ahead.** 'Ongoing' windows (a jam is already in the 60 minutes of history), February odd days (n = 159, never used for any choice) and the 40 official June windows; score = IoU of the predicted queue cells, per window, averaged panel → corridor family → overall.
The tail forecast: the number of jammed links is extrapolated from the last 6 rows (growth g, median 0 here) and the jam edge is moved upstream by g·h links after h steps. LGBM = our gradient-boosted model on the public 36-feature recipe (not reproducible in this notebook), threshold chosen on February even days.

| forecast | Feb odd | June | Δ vs LGBM (Feb odd) [95 %] |
|---|---|---|---|
| persistence (last observed state) | 0.6925 | 0.7484 | −0.097 [−0.123, −0.071] |
| **diagonal tail extrapolation** | **0.6577** | 0.7438 | **−0.124 [−0.156, −0.096]** |
| gradient-boosted model (LGBM) | 0.8008 | 0.7977 | 0 |
| tail OR LGBM | 0.7283 | 0.7640 | −0.066 |
| tail AND LGBM | 0.7301 | 0.7762 | −0.059 |
| persistence OR LGBM | 0.7553 | 0.7894 | −0.048 |

The diagonal extrapolation is *worse than doing nothing* (persistence) - and the model apparently learns the size and decay of a jam from the 36 features by itself (+0.097 over persistence).

**Task 1 - fill a blanked cell.** The same geometry, applied to filling (stage-0 on D7_I10_W, February, ~42,000 cells; the jam row has only 200 cells; error = RMSE of speed km/h / flow per lane veh/h):

| estimator | free flow | in a jam |
|---|---|---|
| linear interpolation in time on the same link | **1.67 / 37** | **4.9 / 50** |
| 'the same cars': single diagonal point (free ±v·5/60 km; jam: wave 20 km/h) | 5.4 / 189 | 51.0 / 235 |
| the whole stretch the cars covered, length-weighted, real km | 3.1 / 121 | 39.7 / 124 |

As **model features** (3 corridors, February, state-score proxy; incumbent 0.9417): + point diagonal and wave diagonal (22 columns) 0.9415, + adaptive diagonal (6) 0.9417, + Lagrangian path (8) 0.9415, + mean over the travelled stretch (4) 0.9420; with all 32 trajectory and group columns 0.9429. The 'same link 5 minutes earlier' is far closer to the truth than 'the same cars 10 km further': each link has its own capacity, free-flow speed and level, so a car's speed is a poor predictor of the speed *there*.

## 5. What did work from this family

One spatial feature won, and it is not a diagonal: a **group window** - the cars that were together at the same time (neighbouring links *within the distance a car covers in 5 minutes*, all at the same time step). Added to the model it gave **0.9417 → 0.9431 (+0.0014)**, and **+0.0008 after blending with a CatBoost** (0.9435 → 0.9443; speed RMSE 1.49 → 1.48, flow per lane 31.69 → 30.94). The information is the *flow* of the group (the number of cars per hour that pass), not the density: the density is flow divided by the local speed, and the speed belongs to the link.

## 6. Why the physics did not transfer - and what we did and did not establish
1. **The speed field is stationary on the link scale** (test 1: best shift 0). Each link has its own level and noise; neighbours differ more than time steps do.
2. **The jam is anchored, and its tail creeps** (test 2, picture): fixed head at the bottleneck, tail growing a few km/h - nowhere near the 24 km/h of the released fundamental diagram. The released parameters describe a diagram, not the process that generated the data.
3. **Not established:** whether real traffic data would behave differently (this is synthetic); the *onset* of a new jam (where the real information is, and which no extrapolation of an existing jam can give); an optimal regime definition (1 km windows and the 0.9/0.6 thresholds are ours).

**Take-aways.** (a) Test the premise of a physical method *before* building the forecast around it: here it was a ten-second experiment on rows of data. (b) A physical principle is a hypothesis about the generator, not about the world; the released parameters (w = 23-28 km/h) contradicted the released data (tail 1-8 km/h). (c) The one spatial feature that paid was found by looking at *what the neighbours share* (cars per hour), not by moving values along a line.

*The static tables in sections 4-5 come from our own measurement log (scripts `t2_diag_tail.py` and `t1_trajectory_feats.py`, not included here); the live cells are self-contained. Data: the competition release only; the live cells run in under a minute. Written with Claude Code.*